## memo

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/notebook?scriptVersionId=344286137
- v3_iid
- answer only adaper

In [ ]:
import polars as pl
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent.parent))
from src.gen_task import RAW_RULES, RAW_OOD_RULES_V1

RULE_CONFIG = RAW_RULES
# RULE_CONFIG = RAW_OOD_RULES_V1

RULES = [
    {
        "id": f"{i:03d}",
        "primitives": rule,
    }
    for i, rule in enumerate(RULE_CONFIG)
]

df = pl.read_csv(Path("debug_predictions.csv"))
df = df.sort("id")
stop = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)


######################################################################################################################################################
Task ID: 
000_0
Prompt: 

Infer the transformation rule from examples.
Output the final array.
        
Example 1

Input:
[1, 0, 4, 3, 3, 2, 1, 8, 1, 9]

Output:
[9, 1, 8, 1, 2, 3, 3, 4, 0, 1]

Example 2

Input:
[0, 0, 1, 3, 3, 8, 9, 0]

Output:
[0, 9, 8, 3, 3, 1, 0, 0]

Example 3

Input:
[3, 8, 6, 3, 7, 9, 4, 0, 2]

Output:
[2, 0, 4, 9, 7, 3, 6, 8, 3]

Example 4

Input:
[6, 5, 4, 2, 3, 5, 1, 1, 6, 1]

Output:
[1, 6, 1, 1, 5, 3, 2, 4, 5, 6]

Example 5

Input:
[5, 9, 4, 0, 7, 8, 1]

Output:
[1, 8, 7, 0, 4, 9, 5]

Query

Input:
[1, 8, 4, 9, 5, 9, 3, 1]

Output:
Raw_output: 
<think>

</think>

[1, 3, 9, 5, 9, 4, 8, 1]
Target: 
[1, 3, 9, 5, 9, 4, 8, 1]
Finish: 
stop
tokens: 
29
######################################################################################################################################################
Task ID: 
000_1

In [6]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match_count = 0
match = []
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match_count += 1
        match.append(pred["id"].split("_")[0])
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match_count: {match_count}")
print(f"acc: {match_count / len(df)}")
# print(miss)
from collections import Counter


counts = Counter(miss)
match_counts = Counter(match)

len(df): 680
match_count: 484
acc: 0.711764705882353


# 正解

In [ ]:
match_results = []

for rule in RULES:
    if rule["id"] in [i for i, j in match_counts.items()]:
        for i, j in match_counts.items():
            if rule["id"] == i:
                match_results.append({"task_id": i, "count": j, "rule": rule["primitives"]})
    else:
        match_results.append({"task_id": rule["id"], "count": 0, "rule": rule["primitives"]})

match_results = sorted(match_results, key=lambda x: x["task_id"], reverse=True)

for x in match_results:
    print(x)

records = []
for x in match_results:
    records.append({"task_id": x["task_id"], "rule": x["rule"], "count": x["count"]})
    print(x)

heatmap_df = pl.DataFrame(records)
pl.Config.set_tbl_rows(-1)

heatmap_df

{'task_id': '071', 'count': 0, 'rule': ['shift_left', 'take_odd_positions']}
{'task_id': '070', 'count': 0, 'rule': ['shift_left', 'take_even_positions']}
{'task_id': '069', 'count': 0, 'rule': ['shift_left', 'mirror']}
{'task_id': '068', 'count': 0, 'rule': ['shift_left', 'adjacent_sum']}
{'task_id': '067', 'count': 9, 'rule': ['shift_left', 'subtract_next']}
{'task_id': '066', 'count': 2, 'rule': ['shift_left', 'mod_3']}
{'task_id': '065', 'count': 1, 'rule': ['shift_left', 'mod_2']}
{'task_id': '064', 'count': 0, 'rule': ['shift_left', 'add_3']}
{'task_id': '063', 'count': 0, 'rule': ['shift_left', 'add_2']}
{'task_id': '062', 'count': 8, 'rule': ['shift_left', 'add_1']}
{'task_id': '061', 'count': 8, 'rule': ['shift_left', 'multiply_3']}
{'task_id': '060', 'count': 9, 'rule': ['shift_left', 'multiply_2']}
{'task_id': '059', 'count': 10, 'rule': ['shift_left', 'pop_left']}
{'task_id': '058', 'count': 10, 'rule': ['shift_left', 'pop_right']}
{'task_id': '057', 'count': 8, 'rule': ['s

# 不正解

In [8]:
results = []
for i, j in counts.items():
    for rule in RULES:
        if rule["id"] == i:
            # print(i, j, rule)
            results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

results = sorted(results, key=lambda x: x["task_id"], reverse=True)
results

[{'task_id': '067', 'count': 1, 'rule': ['shift_left', 'subtract_next']},
 {'task_id': '066', 'count': 8, 'rule': ['shift_left', 'mod_3']},
 {'task_id': '065', 'count': 9, 'rule': ['shift_left', 'mod_2']},
 {'task_id': '064', 'count': 10, 'rule': ['shift_left', 'add_3']},
 {'task_id': '063', 'count': 10, 'rule': ['shift_left', 'add_2']},
 {'task_id': '062', 'count': 2, 'rule': ['shift_left', 'add_1']},
 {'task_id': '061', 'count': 2, 'rule': ['shift_left', 'multiply_3']},
 {'task_id': '060', 'count': 1, 'rule': ['shift_left', 'multiply_2']},
 {'task_id': '057', 'count': 2, 'rule': ['shift_left', 'append_zero']},
 {'task_id': '053', 'count': 3, 'rule': ['shift_right', 'mirror']},
 {'task_id': '052', 'count': 1, 'rule': ['shift_right', 'adjacent_sum']},
 {'task_id': '051', 'count': 2, 'rule': ['shift_right', 'subtract_next']},
 {'task_id': '050', 'count': 10, 'rule': ['shift_right', 'mod_3']},
 {'task_id': '049', 'count': 9, 'rule': ['shift_right', 'mod_2']},
 {'task_id': '047', 'count':